

# Cyclistic Bike-Share Case Study, Part 1: Prepare & Process
**Author:** Makiko Tsuda · **Last updated:** October 2026 · **Data:** Divvy trips, January–December 2025

**The analysis was recorded in two notebook files. This notebook documents the Data Preparation process.**

## Introduction
This is the capstone case study from the Google Data Analytics Professional Certificate. Cyclistic is a fictional bike-share company; the trip data is real, published by Divvy, Chicago’s bike-share system. I analyzed the full 2025 dataset (5.55 million rides; 5.4 million after cleaning) in Python, added daily weather data and OpenStreetMap points of interest, and built an interactive station map in Tableau to test where casual riders concentrate.

### Objective
The director of marketing at Cyclistic believes the company’s future success depends on maximizing the number of annual memberships. I was asked to analyze how casual riders and annual members use Cyclistic bikes differently. The goal is to find insights to design a new marketing strategy to convert casual riders into annual members.

**Business question:** How do annual members and casual riders use Cyclistic bikes differently?

### About this notebook
This notebook prepares the data. It covers two of the six report deliverables:

- **Deliverable 2:** a description of all data sources used
- **Deliverable 3:** documentation of all cleaning and manipulation of the data

- **Input:** 12 monthly trip files in `raw/` (January to December 2025), plus two external sources (weather and points of interest).
- **Output:** `processed/rides_2025_clean.parquet`, the single clean table used by `02_analysis.ipynb`, plus the small external tables.

Run this notebook once, top to bottom. The analysis notebook only reads its outputs.

## 1. Data sources

### Trip data (primary source)

| Item | Detail |
|---|---|
| Source | Divvy trip data, published by Motivate International Inc. ([index of files](https://divvy-tripdata.s3.amazonaws.com/index.html)). Cyclistic is the fictional name used for Divvy in this case study. |
| License | [Divvy Data License Agreement](https://divvybikes.com/data-license-agreement) |
| Period | January 1 to December 31, 2025 (the most recent 12 full months) |
| Format | 12 CSV files, one per month, about 1.1 GB in total |
| Structure | One row per ride, 13 columns: ride ID, bike type, start and end time, start and end station (name and ID), start and end coordinates, rider type (`member` or `casual`) |
| Stored | `raw/` (original files, never modified) |

### Does the data ROCCC?

| Criterion | Assessment |
|---|---|
| **R**eliable | Recorded automatically by the bike system for every ride, not sampled or self-reported. |
| **O**riginal | First-party data published by the bike-share operator. |
| **C**omprehensive | Every ride in 2025, about 5.5 million. Limitation: no rider ID, so repeat riders cannot be tracked. |
| **C**urrent | 2025, the latest full year available. |
| **C**ited | Public source with a published license. |

**Privacy:** the data contains no personal information. Rides cannot be linked to individual riders, so this analysis compares the two rider *types*, not individual behavior. For the same reason, it cannot tell whether casual riders are local residents or visitors.

### External sources (context)

Weather and points-of-interest data were added to explain when and where casual riders ride.

| Source | Used for | Notes |
|---|---|---|
| [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) | Monthly average temperature in Chicago | Free, no API key. Daily mean temperature at 41.88, -87.63 (the Loop). |
| [OpenStreetMap](https://www.openstreetmap.org/copyright) via [Overpass Turbo](https://overpass-turbo.eu) | Attractions, museums, hotels, parks, and Metra commuter rail stations in the downtown Chicago area | © OpenStreetMap contributors (ODbL). Downloaded once as GeoJSON; covers downtown and the lakefront only. |

### Why Python and Jupyter Notebook

I chose Python (pandas) because of the data’s size. The full year has about 5.5 million rides, which is more than Excel’s limit of about 1 million rows per sheet. Python handles the full dataset at once, and every cleaning step is written as code that can be rerun and checked. I used Jupyter Notebook because it keeps the explanation, the code and the charts together in one document, so a reviewer can follow each step from question to result. I built the station map in Tableau Public (web version), where interactive filtering and map layers add the most value.

In [1]:
# Import necessary packages.
import warnings
from pathlib import Path   # used to create paths

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# --- Settings: everything you might change is here ---
data_dir = Path("..")            # project folder, relative to notebooks/
source_folder = "raw"
output_folder = "processed"

ANALYSIS_YEAR = 2025
MIN_DURATION_MIN = 1             # shorter rides are false starts or re-docks, in minutes
MAX_DURATION_MIN = 24 * 60       # longer rides are lost or unreturned bikes, in minutes

# Name of the manually downloaded file from Overpass Turbo (see section 6b for the query)
POI_FILE = "chicago_places.geojson"
# File name for the processed GeoJSON file for Tableau visualization.
VIS_FILE = "places.geojson"

files = sorted((data_dir / source_folder).glob("*.csv"))
print(f"{len(files)} files found")   # should be 12
# List of source data files used for the analysis
files

12 files found


[WindowsPath('../raw/202501-divvy-tripdata.csv'),
 WindowsPath('../raw/202502-divvy-tripdata.csv'),
 WindowsPath('../raw/202503-divvy-tripdata.csv'),
 WindowsPath('../raw/202504-divvy-tripdata.csv'),
 WindowsPath('../raw/202505-divvy-tripdata.csv'),
 WindowsPath('../raw/202506-divvy-tripdata.csv'),
 WindowsPath('../raw/202507-divvy-tripdata.csv'),
 WindowsPath('../raw/202508-divvy-tripdata.csv'),
 WindowsPath('../raw/202509-divvy-tripdata.csv'),
 WindowsPath('../raw/202510-divvy-tripdata.csv'),
 WindowsPath('../raw/202511-divvy-tripdata.csv'),
 WindowsPath('../raw/202512-divvy-tripdata.csv')]

## 2. Load and check the raw data

- Before combining the 12 files, confirm that they all have the same columns. If any month had renamed or reordered columns, combining them would create columns full of missing values.
- Combine them into one dataframe
- Take the summary of the dataset
- Record rider type (member/casual) and bicycle type (electric/classic) split 

In [2]:
column_sets = {f.name: tuple(pd.read_csv(f, nrows=0).columns) for f in files}
pd.Series(column_sets).value_counts()   # one row = every file has the same columns

(ride_id, rideable_type, started_at, ended_at, start_station_name, start_station_id, end_station_name, end_station_id, start_lat, start_lng, end_lat, end_lng, member_casual)    12
Name: count, dtype: int64

In [3]:
# Read each month, tag it with its source file, then combine once.
# The repeated text columns are stored as 'category' to save memory.
frames = []
for f in files:
    df = pd.read_csv(
        f,
        parse_dates=["started_at", "ended_at"],
        dtype={"rideable_type": "category", "member_casual": "category"},
    )
    df["source_file"] = f.stem
    frames.append(df)

# the initial dataframe before cleaning
rides = pd.concat(frames, ignore_index=True)
del frames

# concat can turn category columns back into text when months differ; set them again
for col in ["rideable_type", "member_casual", "source_file"]:
    rides[col] = rides[col].astype("category")

print(f"{len(rides):,} rows, {rides.shape[1]} columns")

5,552,994 rows, 14 columns


In [4]:
# A function that gives a summary of the dataframe
def summarize(df):
    """One row per column: data type, missing values, and distinct values for text columns.

    Blank = empty or whitespace-only text, which isna() does not catch.
    """
    text_cols = df.select_dtypes(include=["object", "category", "string"]).columns

    blanks = pd.Series(0, index=df.columns)
    for c in text_cols:
        blanks[c] = df[c].astype("string").str.strip().eq("").sum()

    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "na_count": df.isna().sum(),
        "blank_count": blanks,
    })
    summary["missing_total"] = summary["na_count"] + summary["blank_count"]
    summary["missing_pct"] = (summary["missing_total"] / len(df) * 100).round(2)
    summary["distinct"] = pd.Series({c: df[c].nunique() for c in text_cols}).astype("Int64")
    return summary

summarize(rides)

,dtype,na_count,blank_count,missing_total,missing_pct,distinct
ride_id,str,0,0,0,0.00,5552994
rideable_type,category,0,0,0,0.00,2
started_at,datetime64[us],0,0,0,0.00,<NA>
ended_at,datetime64[us],0,0,0,0.00,<NA>
start_station_name,str,1184673,0,1184673,21.33,1893
start_station_id,str,1184673,0,1184673,21.33,3360
end_station_name,str,1243305,0,1243305,22.39,1901
end_station_id,str,1243305,0,1243305,22.39,3383
start_lat,float64,0,0,0,0.00,<NA>
start_lng,float64,0,0,0,0.00,<NA>


In [5]:
# Understand rider type and bike type splits
for col in ["rideable_type", "member_casual"]:
    counts = pd.DataFrame({
        "count": rides[col].value_counts(dropna=False),
        "pct": (rides[col].value_counts(normalize=True, dropna=False) * 100).round(1),
    })
    display(counts)

,count,pct
rideable_type,,
electric_bike,3604953,64.9
classic_bike,1948041,35.1


,count,pct
member_casual,,
member,3553497,64.0
casual,1999497,36.0


**What the profile shows:**

- All 12 files share the same 13 columns, so they can be combined safely.
- Rider type and bike type have no missing values.
- About 1 in 5 rides has no start or end station name. These are e-bike rides parked away from a dock (checked below). They are kept: they are valid rides for timing and duration, and simply drop out of station-level counts.
- About 0.1% of rides have no end coordinates. They are also kept for the same reason.
- Station names and station IDs don't match one-to-one, so some IDs carry more than one name (checked below).

## 3. Clean the data

All cleaning happens in this section, in one place, and every step is logged. Each rule and the reason for it:

| Step | Rule | Why |
|---|---|---|
| 1 | Remove duplicate `ride_id` values | Each ride should appear once. |
| 2 | Keep rides that **started** in 2025 | The January file contains a few rides that started on December 31, 2024. |
| 3 | Keep rides lasting 1 minute to 24 hours | Under 1 minute: treated as false starts or bikes re-docked immediately. Over 24 hours: likely lost or unreturned bikes. Both were removed because they would distort averages. |

In [6]:
# Keep track of what cleaning actions were taken
cleaning_log = []

def log_step(step, rows_before, rows_after, reason):
    cleaning_log.append({
        "step": step,
        "rows_before": rows_before,
        "rows_removed": rows_before - rows_after,
        "rows_after": rows_after,
        "reason": reason,
    })
# Start with the original dataframe containing the full dataset from year 2025.
clean = rides

# Step 1: duplicate ride IDs
n = len(clean)
clean = clean.drop_duplicates(subset="ride_id")
log_step("1. Duplicate ride_id", n, len(clean), "Each ride should appear once")

# Step 2: rides that started in the analysis year
n = len(clean)
clean = clean[clean["started_at"].dt.year == ANALYSIS_YEAR]
log_step(f"2. Started in {ANALYSIS_YEAR}", n, len(clean), "January file includes rides from Dec 31, 2024")

# Step 3: plausible duration
duration = (clean["ended_at"] - clean["started_at"]).dt.total_seconds() / 60
n = len(clean)
print("Rides of 0 minutes or less:", (duration <= 0).sum())
print(f"Rides under {MIN_DURATION_MIN} minute:", (duration < MIN_DURATION_MIN).sum())
print(f"Rides over {MAX_DURATION_MIN // 60} hours:", (duration > MAX_DURATION_MIN).sum())
keep = duration.between(MIN_DURATION_MIN, MAX_DURATION_MIN)

# The dataframe after cleaning. This dataset is used for this analysis.
clean = clean[keep].copy()
log_step(f"3. Duration {MIN_DURATION_MIN} min to {MAX_DURATION_MIN // 60} h", n, len(clean),
         "False starts and lost bikes would distort averages")

del rides   # free memory; the raw files are untouched in raw/

log = pd.DataFrame(cleaning_log)
log["pct_removed"] = (log["rows_removed"] / log["rows_before"] * 100).round(2)
log

Rides of 0 minutes or less: 29
Rides under 1 minute: 147401
Rides over 24 hours: 5585


,step,rows_before,rows_removed,rows_after,reason,pct_removed
0,1. Duplicate ride_id,5552994,0,5552994,Each ride should appear once,0.00
1,2. Started in 2025,5552994,53,5552941,"January file includes rides from Dec 31, 2024",0.00
2,3. Duration 1 min to 24 h,5552941,152986,5399955,False starts and lost bikes would distort aver...,2.76


In [7]:
# Count the rides kept and removed by cleaning.
total_removed = cleaning_log[0]["rows_before"] - len(clean)
print(f"Kept {len(clean):,} of {cleaning_log[0]['rows_before']:,} rides "
      f"({len(clean) / cleaning_log[0]['rows_before']:.1%}); removed {total_removed:,}.")

Kept 5,399,955 of 5,552,994 rides (97.2%); removed 153,039.


### Data quality checks (kept, but documented)

These issues don't remove rows, but they affect how some results should be read.

In [8]:
# Rides without a station, by bike type: are they mostly e-bikes?
clean["has_start_station"] = clean["start_station_name"].notna()
clean["has_end_station"] = clean["end_station_name"].notna()

no_station = (~clean["has_start_station"]) | (~clean["has_end_station"])
pd.DataFrame({
    "rides": clean.groupby("rideable_type", observed=True).size(),
    "pct_missing_a_station": (no_station.groupby(clean["rideable_type"], observed=True).mean() * 100).round(1),
})

,rides,pct_missing_a_station
rideable_type,,
classic_bike,1942423,0.0
electric_bike,3457532,50.3


In [9]:
# Station IDs that appear with more than one name (for example, renamed stations)
names_per_id = clean.groupby("start_station_id")["start_station_name"].nunique()
multi_name_ids = names_per_id[names_per_id > 1]
print(f"{len(multi_name_ids)} station IDs have more than one name.")

(clean[clean["start_station_id"].isin(multi_name_ids.index)]
 .groupby(["start_station_id", "start_station_name"]).size()
 .rename("rides").head(10))

78 station IDs have more than one name.


start_station_id  start_station_name           
13017             Franklin St & Chicago Ave        3475
                  Franklin St & Chicago Ave*        176
13154             Sheffield Ave & Kingsbury St     2925
                  Sheffield Ave & Kingsbury St*     102
13192             Halsted St & Dickens Ave         2985
                  Halsted St & Dickens Ave*          94
13208             Burling St & Diversey Pkwy        558
                  Burling St & Diversey Pkwy*        28
13221             Wood St & Milwaukee Ave          3081
                  Wood St & Milwaukee Ave*          106
Name: rides, dtype: int64

**How these are handled in the analysis:**

- Rides without a station are included in all time and duration results, and left out of station rankings and map visualizations. Station results therefore describe docked rides only.
- Stations are grouped by **name**, which is what riders see. The few IDs with more than one name are a small share of rides and don't change the top-station rankings.

## 4. Add analysis columns

Adding `duration_min` (in minutes, easier to average than HH:MM:SS) and `weekday`, plus month, hour and a weekend flag to assist with analysis.

In [10]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

clean["duration_min"] = (clean["ended_at"] - clean["started_at"]).dt.total_seconds() / 60
clean["month"] = clean["started_at"].dt.month
clean["weekday"] = pd.Categorical(clean["started_at"].dt.day_name(), categories=day_order, ordered=True)
clean["hour"] = clean["started_at"].dt.hour
clean["is_weekend"] = clean["weekday"].isin(["Saturday", "Sunday"])

clean[["started_at", "ended_at", "duration_min", "month", "weekday", "hour", "is_weekend", "member_casual"]].head()

,started_at,ended_at,duration_min,month,weekday,hour,is_weekend,member_casual
0,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,13.957950,1,Tuesday,17,False,member
1,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,5.072400,1,Saturday,15,True,member
2,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,11.591667,1,Thursday,15,False,member
3,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,3.570550,1,Thursday,8,False,member
4,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,2.573817,1,Thursday,8,False,member


## 5. Save the clean table

Parquet keeps the data types (dates stay dates, categories stay categories) and loads in seconds. The analysis notebook reads only this file.

In [11]:
clean_file = data_dir / output_folder / f"rides_{ANALYSIS_YEAR}_clean.parquet"
clean.to_parquet(clean_file, index=False)
print(f"Saved {len(clean):,} rows to {clean_file.name}")

Saved 5,399,955 rows to rides_2025_clean.parquet


## 6. Prepare the external data
Weather and points-of-interest data were added to explain when and where casual riders ride.

### 6a. Temperature

Daily mean temperature for 2020 to 2025 from Open-Meteo. The download is saved to `processed/` the first time, so later runs use the saved copy and give the same result.
* Open-Meteo, https://open-meteo.com, is an open-source service that provides weather data through an API.

Two monthly values are kept:
- `temp_2025_f`: the 2025 monthly average, which is compared with 2025 rides.
- `temp_normal_f`: the 2020 to 2025 average, used to check whether 2025 was a typical year.

In [12]:
from urllib.parse import urlencode

temp_start, temp_end = "2020-01-01", f"{ANALYSIS_YEAR}-12-31"
daily_temp_file = data_dir / output_folder / "chicago_daily_temp_2020_2025.csv"

if daily_temp_file.exists():
    weather = pd.read_csv(daily_temp_file, parse_dates=["date"])
else:
    params = {
        "latitude": 41.88,
        "longitude": -87.63,
        "start_date": temp_start,
        "end_date": temp_end,
        "daily": "temperature_2m_mean",
        "temperature_unit": "fahrenheit",
        "timezone": "America/Chicago",
        "format": "csv",
    }
    url = "https://archive-api.open-meteo.com/v1/archive?" + urlencode(params)
    weather = pd.read_csv(url, skiprows=3)   # the first 3 lines describe the location
    weather.columns = ["date", "temp_f"]
    weather["date"] = pd.to_datetime(weather["date"])
    weather.to_csv(daily_temp_file, index=False)

month = weather["date"].dt.month.rename("month")
monthly_temp = pd.DataFrame({
    "temp_2025_f": weather[weather["date"].dt.year == ANALYSIS_YEAR].groupby(month)["temp_f"].mean(),
    "temp_normal_f": weather.groupby(month)["temp_f"].mean(),
}).round(1)
monthly_temp["diff_from_normal_f"] = (monthly_temp["temp_2025_f"] - monthly_temp["temp_normal_f"]).round(1)

monthly_temp.reset_index().to_csv(data_dir / output_folder / "chicago_monthly_temp_summary.csv", index=False)
monthly_temp

,temp_2025_f,temp_normal_f,diff_from_normal_f
month,,,
1,22.3,26.4,-4.1
2,26.2,28.2,-2.0
3,42.9,40.0,2.9
4,49.5,48.2,1.3
5,55.5,58.2,-2.7
6,71.8,70.8,1.0
7,76.2,74.0,2.2
8,72.7,73.2,-0.5
9,67.5,67.4,0.1


### 6b. Points of interest and commuter rail stations

* OpenStreetMap (OSM) is a free, editable map of the world, built and maintained by contributors.
The coordinates for the points of interest and Metra commuter rail stations in the downtown Chicago area were downloaded once in the browser from [Overpass Turbo](https://overpass-turbo.eu) (the public API does not accept requests from scripts) and saved as `processed/chicago_places.geojson`. 


The query covers downtown and the lakefront:

```
[out:json][timeout:120];
(
  nwr["tourism"~"^(attraction|museum|hotel|viewpoint)$"](41.83,-87.70,41.97,-87.58);
  nwr["leisure"="park"](41.83,-87.70,41.97,-87.58);
  nwr["railway"="station"]["network"~"Metra"](41.83,-87.70,41.97,-87.58);
);
out center tags;
```

Places that link to Wikidata are flagged as `notable` (well-known landmarks and venues). Metra stations are always kept, so they appear on the map whether or not they link to Wikidata.

In [13]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")      # GDAL setup warning on some Windows installs
    import geopandas as gpd
    pois = gpd.read_file(data_dir / output_folder / POI_FILE)

# Parks and buildings are outlines; use their center point.
# Centers are computed in meters (UTM zone 16N), then converted back to latitude/longitude.
centers = pois.geometry.to_crs(epsg=26916).centroid.to_crs(epsg=4326)
pois["lat"] = centers.y
pois["lng"] = centers.x

# A tag column only exists if at least one downloaded place has that tag
def tag(name):
    return pois[name] if name in pois else pd.Series(pd.NA, index=pois.index)

is_rail = tag("railway").eq("station")

pois["category"] = (tag("tourism")
                    .fillna(tag("amenity"))
                    .fillna(tag("leisure")))
pois.loc[is_rail, "category"] = "commuter rail"

# Notable = linked to Wikidata; rail stations are always kept on the map
pois["notable"] = tag("wikidata").notna() | is_rail

places = (pois[["name", "category", "notable", "lat", "lng"]]
          .dropna(subset=["name"])                                # unnamed places are usually minor
          .drop_duplicates(subset=["name", "category"])           # a station can be both a point and an outline
          .assign(layer="Place"))
places.to_csv(data_dir / output_folder / "places.csv", index=False)

# Saving the processed GeoJSON file to upload it to Tableau.
places_geo = gpd.GeoDataFrame(
    places,
    geometry=gpd.points_from_xy(places["lng"], places["lat"]),
    crs="EPSG:4326",            # standard latitude/longitude
)
places_geo.to_file(data_dir / output_folder / VIS_FILE, driver="GeoJSON")

places.groupby("category").agg(places=("name", "size"), notable=("notable", "sum")).sort_values("places", ascending=False)

,places,notable
category,,
park,253,101
hotel,164,27
attraction,72,8
museum,43,31
commuter rail,11,11
viewpoint,8,0


## Summary of outputs

| File in `processed/` | Contents | Used by |
|---|---|---|
| `rides_2025_clean.parquet` | Clean ride table with analysis columns | `02_analysis.ipynb` |
| `chicago_daily_temp_2020_2025.csv` | Saved daily temperature download | This notebook (cache) |
| `chicago_monthly_temp_summary.csv` | 2025 and 2020–2025 monthly averages | `02_analysis.ipynb` |
| `places.csv` | Named points of interest with category and notable flag | `02_analysis.ipynb` |
| `places.geojson` | Place locations for the Tableau map | Tableau |